# Лабораторная работа #1
## Конструктивное число и методы оптимизации

In [ ]:
import sys
sys.path.append('src')

import random
import numpy as np

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)

seed_everything()

import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120

from constructive_number import ConstructiveNumber
from black_box import make_quadratic_good, make_quadratic_bad, make_rosenbrock
from optimizers import NelderMead, GradientDescent, NewtonMethod

## 1. Конструктивное число — проверка арифметики

In [ ]:
# Создание из действительного числа
a = ConstructiveNumber.from_real(3.14159, eps=0.01)
b = ConstructiveNumber.from_real(2.71828, eps=0.01)
print('a =', a)
print('b =', b)

# Арифметика
print('a + b =', a + b)
print('a - b =', a - b)
print('a * b =', a * b)
print('a / b =', a / b)

# Получение значения через alpha
print(f'a.get(alpha=0)   = {a.get(0):.6f}  (нижняя граница)')
print(f'a.get(alpha=0.5) = {a.get(0.5):.6f}  (середина)')
print(f'a.get(alpha=1)   = {a.get(1):.6f}  (верхняя граница)')

# Создание из пары рациональных чисел
c = ConstructiveNumber.from_pair(1, 2)
print('c =', c, ', eps =', c.eps)

## 2. Рост ε при арифметических операциях

In [ ]:
eps0 = 1e-3
x = ConstructiveNumber.from_real(2.0, eps0)

eps_history = [x.eps]
labels = ['x']
current = x

for i in range(1, 15):
    current = current * x  # x^(i+1)
    eps_history.append(current.eps)
    labels.append(f'x^{i+1}')

plt.figure(figsize=(9, 4))
plt.plot(range(1, len(eps_history)+1), eps_history, 'o-', color='steelblue')
plt.xticks(range(1, len(labels)+1), labels, rotation=45)
plt.yscale('log')
plt.ylabel('ε (ширина интервала)')
plt.title('Рост ε при последовательном умножении x на себя (начальный ε=1e-3)')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('eps_growth_arithmetic.png', dpi=120)
plt.show()

## 3. Функции (черные ящики) — визуализация

In [ ]:
# Визуализация линий уровня для 2D среза каждой функции
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

def plot_contour(ax, f, xlim, ylim, fixed_dims, title, n=200):
    xs = np.linspace(*xlim, n)
    ys = np.linspace(*ylim, n)
    Z = np.zeros((n, n))
    for i, xi in enumerate(xs):
        for j, yj in enumerate(ys):
            pt = list(fixed_dims)
            pt[0] = xi
            pt[1] = yj
            Z[j, i] = float(f(pt))
    ax.contourf(xs, ys, Z, levels=30, cmap='viridis')
    ax.contour(xs, ys, Z, levels=15, colors='white', linewidths=0.5, alpha=0.5)
    ax.set_title(title)
    ax.set_xlabel('x₀')
    ax.set_ylabel('x₁')

f_good = make_quadratic_good()
f_bad = make_quadratic_bad()
f_ros = make_rosenbrock()

plot_contour(axes[0], f_good, (-3, 3), (-3, 3), [0.0]*6, 'Квадратичная 6D\n(хорошая обусловленность)')
plot_contour(axes[1], f_bad,  (-3, 3), (-3, 3), [0.0]*4, 'Квадратичная 4D\n(плохая обусловленность)')
plot_contour(axes[2], f_ros,  (-2, 2), (-1, 3), [0.0, 0.0, 1.0], 'Розенброк 3D\n(срез x₂=1)')

plt.tight_layout()
plt.savefig('contour_plots.png', dpi=120)
plt.show()

## 4. Оптимизация — float (без CN)

In [ ]:
functions = [
    (make_quadratic_good, [0.5]*6, 'Quadratic-6D-good'),
    (make_quadratic_bad,  [0.5]*4, 'Quadratic-4D-bad'),
    (make_rosenbrock,     [-1.0, 1.0, -0.5], 'Rosenbrock-3D'),
]

results = {}

for make_f, x0, fname in functions:
    results[fname] = {}
    for OptimizerClass, kwargs, oname in [
        (NelderMead,       dict(tol=1e-8, max_iter=20000), 'Nelder-Mead'),
        (GradientDescent,  dict(lr=0.1, tol=1e-8, max_iter=10000, line_search=True), 'GradDesc'),
        (NewtonMethod,     dict(tol=1e-8, max_iter=500), 'Newton'),
    ]:
        f = make_f()
        opt = OptimizerClass(**kwargs)
        xstar, history = opt.minimize(f, x0)
        results[fname][oname] = {
            'xstar': xstar,
            'fstar': history[-1]['f'],
            'iters': len(history),
            'f_calls': f.f_calls,
            'grad_calls': f.grad_calls,
            'hess_calls': f.hess_calls,
            'history': history,
        }
        print(f'{fname} | {oname}: f*={history[-1]["f"]:.2e}, iters={len(history)}, f_calls={f.f_calls}')

## 5. Траектории спуска на линиях уровня

In [ ]:
def plot_trajectory(ax, history, label, color, max_pts=300):
    xs = [h['x'][0] for h in history[:max_pts]]
    ys = [h['x'][1] for h in history[:max_pts]]
    ax.plot(xs, ys, '-o', color=color, markersize=2, linewidth=1, label=label, alpha=0.8)
    ax.plot(xs[0], ys[0], 's', color=color, markersize=6)
    ax.plot(xs[-1], ys[-1], '*', color=color, markersize=10)

colors = {'Nelder-Mead': 'tomato', 'GradDesc': 'royalblue', 'Newton': 'seagreen'}

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

contour_configs = [
    ('Quadratic-6D-good', make_quadratic_good, (-1,1), (-1,1), [0.0]*6, 'Quadratic-6D (срез x₂..x₅=0)'),
    ('Quadratic-4D-bad',  make_quadratic_bad,  (-1,1), (-1,1), [0.0]*4, 'Quadratic-4D bad cond'),
    ('Rosenbrock-3D',     make_rosenbrock,     (-2,2), (-1,3), [0.0,0.0,1.0], 'Rosenbrock (срез x₂=1)'),
]

for ax, (fname, make_f, xlim, ylim, fixed, title) in zip(axes, contour_configs):
    f_tmp = make_f()
    xs = np.linspace(*xlim, 150)
    ys = np.linspace(*ylim, 150)
    Z = np.zeros((150, 150))
    for i, xi in enumerate(xs):
        for j, yj in enumerate(ys):
            pt = list(fixed)
            pt[0] = xi; pt[1] = yj
            Z[j, i] = float(f_tmp(pt))
    ax.contourf(xs, ys, Z, levels=25, cmap='Blues_r', alpha=0.7)
    ax.contour(xs, ys, Z, levels=15, colors='gray', linewidths=0.4, alpha=0.6)

    for oname, color in colors.items():
        history = results[fname][oname]['history']
        plot_trajectory(ax, history, oname, color)

    ax.set_title(title)
    ax.set_xlabel('x₀'); ax.set_ylabel('x₁')
    ax.legend(fontsize=8)

plt.suptitle('Траектории методов оптимизации', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('trajectories.png', dpi=120)
plt.show()

## 6. Сходимость: f(iter) для каждого метода

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

for ax, (fname, _, __, ___, ____, title) in zip(axes, contour_configs):
    for oname, color in colors.items():
        h = results[fname][oname]['history']
        fvals = [max(d['f'], 1e-16) for d in h]
        ax.semilogy(fvals, color=color, label=oname, linewidth=1.5)
    ax.set_title(title)
    ax.set_xlabel('Итерации')
    ax.set_ylabel('f(x) (log)')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.suptitle('Скорость сходимости', fontsize=13)
plt.tight_layout()
plt.savefig('convergence.png', dpi=120)
plt.show()

## 7. Оптимизация с Конструктивным числом — исследование ε

In [ ]:
eps_values = [1e-2, 1e-4, 1e-6]
eps_results = {}

test_cases = [
    (make_quadratic_bad, [0.5]*4, 'Quadratic-4D-bad'),
    (make_rosenbrock,    [-1.0, 1.0, -0.5], 'Rosenbrock-3D'),
]

for make_f, x0, fname in test_cases:
    eps_results[fname] = {}
    for eps in eps_values:
        eps_results[fname][eps] = {}
        for OptimizerClass, kwargs, oname in [
            (NelderMead,      dict(tol=1e-6, max_iter=10000, cn_eps=eps), 'Nelder-Mead'),
            (GradientDescent, dict(lr=0.1, tol=1e-6, max_iter=5000, line_search=True, cn_eps=eps), 'GradDesc'),
            (NewtonMethod,    dict(tol=1e-6, max_iter=500, cn_eps=eps), 'Newton'),
        ]:
            f = make_f()
            opt = OptimizerClass(**kwargs)
            xstar, history = opt.minimize(f, x0)
            eps_results[fname][eps][oname] = {
                'fstar': history[-1]['f'],
                'iters': len(history),
                'history': history,
            }
            print(f'{fname} | ε={eps:.0e} | {oname}: f*={history[-1]["f"]:.4e}, iters={len(history)}')

## 8. Динамика ε в процессе оптимизации

In [ ]:
# Для Rosenbrock — показываем как eps влияет на сходимость
fname = 'Rosenbrock-3D'

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

for ax, oname in zip(axes, ['Nelder-Mead', 'GradDesc', 'Newton']):
    for eps, lcolor in zip(eps_values, ['tomato', 'royalblue', 'seagreen']):
        h = eps_results[fname][eps][oname]['history']
        fvals = [max(d['f'], 1e-16) for d in h]
        ax.semilogy(fvals, color=lcolor, label=f'ε={eps:.0e}', linewidth=1.5)
    ax.set_title(f'{oname}\n(Rosenbrock)')
    ax.set_xlabel('Итерации')
    ax.set_ylabel('f(x) (log)')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.suptitle('Влияние начального ε на сходимость (Rosenbrock)', fontsize=13)
plt.tight_layout()
plt.savefig('eps_convergence.png', dpi=120)
plt.show()

## 9. Сравнение ε по функциям и методам

In [ ]:
# Таблица: итоговое f* при разных eps и методах
import pandas as pd

rows = []
for fname in eps_results:
    for eps in eps_values:
        for oname in ['Nelder-Mead', 'GradDesc', 'Newton']:
            r = eps_results[fname][eps][oname]
            rows.append({
                'Function': fname,
                'ε': f'{eps:.0e}',
                'Method': oname,
                'f*': f"{r['fstar']:.4e}",
                'Iters': r['iters'],
            })

df = pd.DataFrame(rows)
print(df.to_string(index=False))

## 10. Итоги по вызовам функции

In [ ]:
# Количество вызовов f и grad для float-версии
fig, ax = plt.subplots(figsize=(10, 5))

method_names = list(colors.keys())
x_pos = np.arange(len(functions))
width = 0.25

for i, oname in enumerate(method_names):
    f_calls = [results[fname][oname]['f_calls'] for _, __, fname in functions]
    ax.bar(x_pos + i*width, f_calls, width, label=oname, color=list(colors.values())[i], alpha=0.8)

ax.set_xticks(x_pos + width)
ax.set_xticklabels([fname for _, __, fname in functions])
ax.set_ylabel('Количество вызовов f')
ax.set_title('Количество вызовов функции по методам')
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('f_calls.png', dpi=120)
plt.show()